# Notebook 5: 04_metrics_and_threshold
Metrics, PR curve, threshold trade-off, and cost estimation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
FIG_DIR = PROJECT_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_PATH = DATA_DIR / 'audio_clip_event_cleaned.csv'
if not AUDIO_PATH.exists():
    AUDIO_PATH = DATA_DIR / 'audio_clip_event.csv'

df = pd.read_csv(AUDIO_PATH)
y_true = df['binary_label'].astype(int).to_numpy()
score = df['model_score'].to_numpy()
print('Using:', AUDIO_PATH)
print('Total samples:', len(df))


In [ ]:
def metrics_at_threshold(y_true, score, threshold):
    pred = (score >= threshold).astype(int)
    tp = int(((y_true==1)&(pred==1)).sum())
    fp = int(((y_true==0)&(pred==1)).sum())
    fn = int(((y_true==1)&(pred==0)).sum())
    tn = int(((y_true==0)&(pred==0)).sum())
    accuracy = (tp + tn) / len(y_true)
    precision = tp/(tp+fp) if (tp+fp) else 0.0
    recall = tp/(tp+fn) if (tp+fn) else 0.0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0
    review_cost = (tp + fp) * 8
    false_alarm_cost = fp * 80
    miss_cost = fn * 300
    total_cost = review_cost + false_alarm_cost + miss_cost
    return {
        'threshold': threshold,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1,
        'alert_volume': tp + fp,
        'review_cost': review_cost,
        'false_alarm_cost': false_alarm_cost,
        'miss_cost': miss_cost,
        'estimated_total_cost': total_cost,
    }


In [ ]:
# Baseline at threshold 0.8
pd.DataFrame([metrics_at_threshold(y_true, score, 0.80)])[['threshold','tp','fp','fn','tn','accuracy','precision','recall','f1']]


In [ ]:
# PR curve
sweep = np.linspace(0.0, 1.0, 201)
pr = pd.DataFrame([metrics_at_threshold(y_true, score, t) for t in sweep])
plt.figure(figsize=(6,5))
plt.plot(pr['recall'], pr['precision'], color='#2563eb', linewidth=2)
plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(FIG_DIR/'pr_curve.png', dpi=150)
plt.show()


In [ ]:
# Threshold vs precision / recall
ts = np.linspace(0.50, 0.95, 91)
tr = pd.DataFrame([metrics_at_threshold(y_true, score, t) for t in ts])
plt.figure(figsize=(8,4.5))
plt.plot(tr['threshold'], tr['precision'], label='Precision', linewidth=2, color='#16a34a')
plt.plot(tr['threshold'], tr['recall'], label='Recall', linewidth=2, color='#dc2626')
plt.title('Threshold vs Precision / Recall')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR/'threshold_tradeoff.png', dpi=150)
plt.show()


In [ ]:
# Cost table for requested thresholds
check = [0.60, 0.70, 0.80, 0.85, 0.90]
cost_df = pd.DataFrame([metrics_at_threshold(y_true, score, t) for t in check])
cost_df[['threshold','precision','recall','f1','alert_volume','fp','fn','review_cost','false_alarm_cost','miss_cost','estimated_total_cost']]


In [ ]:
# Optional 0.86 reference
pd.DataFrame([metrics_at_threshold(y_true, score, 0.86)])[['threshold','precision','recall','f1','alert_volume','fp','fn','estimated_total_cost']]
